# Roihu – harjoitus 3: Rakenna itse

Kahdessa ensimmäisessä harjoituksessa koodi oli valmiina:

- [cifar10_roihu.ipynb](cifar10_roihu.ipynb) – Roihun GPU-ympäristön testaus ja ensimmäinen CNN
- [cifar10_roihu_2.ipynb](cifar10_roihu_2.ipynb) – mallin parantaminen ja tulosten analysointi

Tässä harjoituksessa rakennat saman koneoppimisen työketjun kuin harjoituksessa 2,
mutta kirjoitat sen keskeiset osat itse.

## Miten harjoitus toimii?

Tehtävät 1–7 etenevät samalla kaavalla. Ensin tulee tehtävän kuvaus ja vihje, sitten
koodisolu, jossa on `TODO`-kommentteja täydennettäväksi. Sen alla on piilotettu
malliratkaisu; avaa se vasta, kun olet yrittänyt itse tai jäät jumiin. Lopuksi tulee
tarkistussolu, joka testaa koodisi ja kertoo, mikä meni pieleen.

Älä siirry seuraavaan tehtävään ennen kuin edellisen tarkistus menee läpi, koska
myöhemmät tehtävät käyttävät aiempien tehtävien koodia.

Tehtävä 8 on haaste, jossa parannat mallia omin päin. Siinä ei ole vihjettä,
malliratkaisua eikä tarkistusta.

## Tehtävät

| # | Tehtävä | Aihe |
| --- | --- | --- |
| 1 | Normalisointiarvot | NumPy, datan tilastot |
| 2 | Muunnokset | Normalisointi ja augmentaatio |
| 3 | Datan jako | Train / validation |
| 4 | Oma CNN | `nn.Module`, `nn.Sequential` |
| 5 | Koulutusaskel | Forward, loss, backward, optimizer |
| 6 | Arviointi | `model.eval()`, `torch.no_grad()` |
| 7 | Confusion matrix | Tulosten analysointi |
| 8 | Haaste | Paranna mallia itse |

Käytä samoja Jupyter-asetuksia kuin aiemmissa harjoituksissa
(`gpuinteractive`, 4 CPU-ydintä, 16 GiB, `python-pytorch/2.13`).

## 0. Valmistelut

Seuraava solu tuo kirjastot, valitsee laskentalaitteen ja määrittelee apufunktion
`check`, jota tarkistussolut käyttävät. Siihen ei tarvitse koskea; aja se sellaisenaan.

In [ ]:
import time
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU ei ole käytettävissä.")


def check(condition, message):
    if condition:
        print("✅", message)
    else:
        raise AssertionError("❌ " + message)

## Tehtävä 1: Normalisointiarvot

Harjoituksessa 2 opittiin, että neuroverkko oppii paremmin, kun jokaisen
värikanavan keskiarvo on noin 0 ja keskihajonta noin 1.

Tätä varten tarvitaan koulutusdatan keskiarvo ja keskihajonta jokaiselle
värikanavalle (R, G, B).

Alla `pixels` on NumPy-taulukko, jonka muoto on:

`(50000, 32, 32, 3)` = (kuvat, korkeus, leveys, värikanavat)

### Tehtävä

Laske `MEAN` ja `STD` niin, että kumpikin on kolmen luvun tuple, yksi luku
jokaiselle värikanavalle.

<details>
<summary>Vihje</summary>

NumPy-taulukon `mean()` ja `std()` ottavat parametrin `axis`, joka kertoo, minkä
akselien yli lasketaan. Haluat laskea kaikkien akselien yli paitsi viimeisen
(värikanavat), eli akselien `0`, `1` ja `2` yli.

NumPy-taulukon saa listaksi `.tolist()`-metodilla ja listan tupleksi `tuple()`-funktiolla.

</details>

In [ ]:
raw_trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True
)

pixels = raw_trainset.data / 255.0

# TODO: laske jokaisen värikanavan keskiarvo ja keskihajonta
MEAN = None
STD = None

print("Mean:", MEAN)
print("Std: ", STD)

<details>
<summary>Malliratkaisu (avaa vasta, kun olet yrittänyt itse)</summary>

```python
raw_trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True
)

pixels = raw_trainset.data / 255.0

MEAN = tuple(pixels.mean(axis=(0, 1, 2)).tolist())
STD = tuple(pixels.std(axis=(0, 1, 2)).tolist())

print("Mean:", MEAN)
print("Std: ", STD)
```

</details>

### Tarkistus 1

In [ ]:
check(MEAN is not None and STD is not None, "MEAN ja STD on asetettu")
check(len(MEAN) == 3 and len(STD) == 3, "MEAN ja STD sisältävät kolme arvoa")

expected_mean = (0.4914, 0.4822, 0.4465)
expected_std = (0.2470, 0.2435, 0.2616)

check(
    all(abs(m - e) < 0.01 for m, e in zip(MEAN, expected_mean)),
    "MEAN on oikein"
)
check(
    all(abs(s - e) < 0.01 for s, e in zip(STD, expected_std)),
    "STD on oikein"
)

## Tehtävä 2: Muunnokset

Seuraavaksi määritellään muunnokset (`transforms`), jotka tehdään jokaiselle kuvalle.

### Tehtävä

Täydennä kaksi muunnosta.

`train_transform` koulutusdatalle:

1. satunnainen rajaus: `RandomCrop`, kuvan koko 32 ja `padding=4`
2. satunnainen vaakapeilaus: `RandomHorizontalFlip`
3. `ToTensor()` (valmiina)
4. normalisointi: `Normalize` käyttäen tehtävän 1 arvoja

`test_transform` validointi- ja testidatalle:

1. `ToTensor()` (valmiina)
2. normalisointi

Validointi- ja testidataan ei käytetä augmentaatiota.

<details>
<summary>Vihje</summary>

Augmentaatiot tehdään PIL-kuvalle, joten niiden pitää olla ennen `ToTensor()`-muunnosta.
`Normalize` taas toimii tensorille, joten sen pitää olla sen jälkeen.

Katso tarvittaessa harjoituksen 2 kohta 3.

</details>

In [ ]:
train_transform = transforms.Compose([
    # TODO: satunnainen rajaus ja vaakapeilaus

    transforms.ToTensor(),

    # TODO: normalisointi
])

test_transform = transforms.Compose([
    transforms.ToTensor(),

    # TODO: normalisointi
])

<details>
<summary>Malliratkaisu (avaa vasta, kun olet yrittänyt itse)</summary>

```python
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])
```

</details>

### Tarkistus 2

In [ ]:
def contains(transform, kind):
    return any(isinstance(t, kind) for t in transform.transforms)


check(contains(train_transform, transforms.RandomCrop), "train_transform sisältää RandomCropin")
check(contains(train_transform, transforms.RandomHorizontalFlip), "train_transform sisältää RandomHorizontalFlipin")
check(contains(train_transform, transforms.Normalize), "train_transform sisältää normalisoinnin")
check(contains(test_transform, transforms.Normalize), "test_transform sisältää normalisoinnin")
check(
    not contains(test_transform, transforms.RandomCrop)
    and not contains(test_transform, transforms.RandomHorizontalFlip),
    "test_transform ei sisällä augmentaatiota"
)

sample = torch.stack([test_transform(raw_trainset[i][0]) for i in range(1000)])

check(sample.shape[1:] == (3, 32, 32), "Kuvan muoto on (3, 32, 32)")
check(abs(sample.mean().item()) < 0.1, f"Keskiarvo on lähellä nollaa ({sample.mean():.3f})")
check(abs(sample.std().item() - 1) < 0.1, f"Keskihajonta on lähellä ykköstä ({sample.std():.3f})")

## Tehtävä 3: Datan jako

Jaetaan 50 000 koulutuskuvaa kahteen osaan: 45 000 kuvaa koulutukseen (`train`)
ja 5 000 kuvaa validointiin (`validation`).

Alla `indices` on lista, jossa luvut 0–49 999 ovat satunnaisessa järjestyksessä.

### Tehtävä

Jaa `indices` kahteen osaan:

- `val_indices` – listan ensimmäiset `VAL_SIZE` indeksiä
- `train_indices` – loput indeksit

Saman kuvan ei pidä päätyä molempiin osiin.

<details>
<summary>Vihje</summary>

Pythonin listan voi pilkkoa näin:

```python
luvut = [10, 20, 30, 40, 50]
luvut[:2]   # [10, 20]
luvut[2:]   # [30, 40, 50]
```

</details>

In [ ]:
VAL_SIZE = 5000

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(raw_trainset), generator=generator).tolist()

# TODO: jaa indeksit validointi- ja koulutusosaan
val_indices = None
train_indices = None

<details>
<summary>Malliratkaisu (avaa vasta, kun olet yrittänyt itse)</summary>

```python
VAL_SIZE = 5000

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(raw_trainset), generator=generator).tolist()

val_indices = indices[:VAL_SIZE]
train_indices = indices[VAL_SIZE:]
```

</details>

### Tarkistus 3

In [ ]:
check(val_indices is not None and train_indices is not None, "Indeksit on asetettu")
check(len(val_indices) == 5000, "Validointiosassa on 5 000 kuvaa")
check(len(train_indices) == 45000, "Koulutusosassa on 45 000 kuvaa")
check(len(set(val_indices) & set(train_indices)) == 0, "Osissa ei ole yhteisiä kuvia")

### Datasetit ja DataLoaderit

Kun tarkistus menee läpi, aja seuraava solu. Se luo datasetit ja DataLoaderit
samalla tavalla kuin harjoituksessa 2.

In [ ]:
full_train_augmented = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transform
)
full_train_plain = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=test_transform
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=test_transform
)

trainset = Subset(full_train_augmented, train_indices)
valset = Subset(full_train_plain, val_indices)
classes = raw_trainset.classes

BATCH_SIZE = 128

trainloader = DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
valloader = DataLoader(
    valset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)
testloader = DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

print("Training images:  ", len(trainset))
print("Validation images:", len(valset))
print("Test images:      ", len(testset))

## Tehtävä 4: Oma CNN

Nyt rakennetaan malli kahdessa osassa.

### Osa A: `conv_block`

Kirjoita funktio `conv_block(in_channels, out_channels)`, joka palauttaa
`nn.Sequential`-lohkon, jossa on järjestyksessä:

1. `nn.Conv2d` – kernelin koko 3, `padding=1`, `bias=False`
2. `nn.BatchNorm2d`
3. `nn.ReLU`

`padding=1` pitää kuvan koon samana. Esimerkiksi 32 × 32 → 32 × 32.

### Osa B: `MyCNN`

Rakenna `self.features` kolmesta lohkosta. Jokaisessa lohkossa on kaksi
`conv_block`-kerrosta, `nn.MaxPool2d(2)` joka puolittaa kuvan koon, ja `nn.Dropout`.

| Lohko | Kanavat sisään → ulos | Kuvan koko | Dropout |
| --- | --- | --- | --- |
| 1 | 3 → 64 | 32 → 16 | 0.1 |
| 2 | 64 → 128 | 16 → 8 | 0.2 |
| 3 | 128 → 256 | 8 → 4 | 0.3 |

`self.classifier` on valmiina. Se odottaa, että `self.features` palauttaa tensorin
muotoa `(batch, 256, 4, 4)`.

Täydennä lopuksi `forward`-metodi: syöte kulkee ensin `self.features`-osan ja
sitten `self.classifier`-osan läpi.

<details>
<summary>Vihje</summary>

Ensimmäisen lohkon kaksi kerrosta ovat `conv_block(3, 64)` ja `conv_block(64, 64)`.
Toinen kerros ottaa siis sisäänsä saman määrän kanavia kuin ensimmäinen tuottaa.

Harjoituksen 2 kohdassa 5 on lähes sama malli.

</details>

In [ ]:
def conv_block(in_channels, out_channels):
    # TODO: palauta nn.Sequential, jossa on Conv2d, BatchNorm2d ja ReLU
    return None


class MyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # TODO: kolme lohkoa, joissa kussakin kaksi conv_blockia,
        #       MaxPool2d ja Dropout
        self.features = nn.Sequential(

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # TODO: kuljeta x ensin self.featuresin ja sitten self.classifierin läpi
        return x

<details>
<summary>Malliratkaisu (avaa vasta, kun olet yrittänyt itse)</summary>

```python
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )


class MyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            conv_block(3, 64),
            conv_block(64, 64),
            nn.MaxPool2d(2),
            nn.Dropout(0.1),

            conv_block(64, 128),
            conv_block(128, 128),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            conv_block(128, 256),
            conv_block(256, 256),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
```

</details>

### Tarkistus 4

In [ ]:
block = conv_block(3, 16)
check(isinstance(block, nn.Sequential), "conv_block palauttaa nn.Sequentialin")
check(
    any(isinstance(layer, nn.BatchNorm2d) for layer in block),
    "conv_block sisältää BatchNorm2d-kerroksen"
)
check(
    block(torch.randn(2, 3, 32, 32)).shape == (2, 16, 32, 32),
    "conv_block säilyttää kuvan koon ja muuttaa kanavien määrän"
)

test_model = MyCNN()
dummy = torch.randn(8, 3, 32, 32)

check(
    test_model.features(dummy).shape == (8, 256, 4, 4),
    "self.features palauttaa muodon (8, 256, 4, 4)"
)
check(test_model(dummy).shape == (8, 10), "Malli palauttaa muodon (8, 10)")
check(
    any(isinstance(layer, nn.Dropout) for layer in test_model.features),
    "self.features sisältää Dropoutin"
)

num_parameters = sum(p.numel() for p in test_model.parameters())
print(f"Parameters: {num_parameters:,}")

## Tehtävä 5: Koulutusaskel

Jokaisella batchilla tehdään samat viisi vaihetta:

1. Nollataan gradientit, koska PyTorch muuten laskee ne edellisten batchien päälle
2. Forward pass: malli tekee ennusteen
3. Loss: ennustetta verrataan oikeisiin vastauksiin
4. Backward pass: lasketaan gradientit eli se, mihin suuntaan painoja pitää muuttaa
5. Optimizer step: päivitetään painot

### Tehtävä

Kirjoita funktio `train_step`, joka tekee nämä viisi vaihetta yhdelle batchille ja
palauttaa `loss`- ja `outputs`-arvot.

Funktio `train_one_epoch` on valmiina. Se kutsuu `train_step`-funktiota jokaiselle
batchille.

<details>
<summary>Vihje</summary>

Tarvitset nämä viisi riviä, mutta oikeassa järjestyksessä:

```python
loss.backward()
outputs = model(images)
optimizer.step()
optimizer.zero_grad()
loss = criterion(outputs, labels)
```

</details>

In [ ]:
criterion = nn.CrossEntropyLoss()


def train_step(model, images, labels, optimizer):
    # TODO 1: nollaa gradientit

    # TODO 2: forward pass
    outputs = None

    # TODO 3: laske loss
    loss = None

    # TODO 4: backward pass

    # TODO 5: päivitä painot

    return loss, outputs

<details>
<summary>Malliratkaisu (avaa vasta, kun olet yrittänyt itse)</summary>

```python
criterion = nn.CrossEntropyLoss()


def train_step(model, images, labels, optimizer):
    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(outputs, labels)

    loss.backward()

    optimizer.step()

    return loss, outputs
```

</details>

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        loss, outputs = train_step(model, images, labels, optimizer)

        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100 * correct / total

### Tarkistus 5: "overfit one batch"

Ennen pitkää koulutusta kannattaa varmistaa, että koulutuskoodi ylipäätään toimii.

Alla otetaan 64 kuvaa ja koulutetaan mallia pelkästään niillä 150 kierrosta. Jos
koulutuskoodi toimii, malli oppii nämä kuvat lähes ulkoa ja loss putoaa lähelle
nollaa.

Jos loss ei pienene, koodissa on virhe. Silloin pitkä koulutus olisi turha.

In [ ]:
torch.manual_seed(SEED)

test_model = MyCNN().to(device)
test_optimizer = optim.Adam(test_model.parameters(), lr=0.001)
test_model.train()

images, labels = next(iter(trainloader))
images = images[:64].to(device)
labels = labels[:64].to(device)

losses = []

for step in range(150):
    loss, outputs = train_step(test_model, images, labels, test_optimizer)

    if loss is None or outputs is None:
        check(False, "train_step palauttaa lossin ja ennusteet")

    losses.append(loss.item())

print(f"Loss alussa: {losses[0]:.3f}")
print(f"Loss lopussa: {losses[-1]:.3f}")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.title("Overfit one batch")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.show()

check(losses[-1] < losses[0], "Loss pienenee")
check(losses[-1] < 0.5, "Malli oppii yhden batchin lähes ulkoa")

## Tehtävä 6: Arviointi

Kirjoita funktio `evaluate`, joka laskee mallin keskimääräisen lossin ja
tarkkuuden (`accuracy`, prosentteina) annetulla datalla.

### Tehtävä

Täydennä `TODO`-kohdat:

1. Aseta malli arviointitilaan. Tämä kytkee Dropoutin pois ja asettaa BatchNormin
   käyttämään koulutuksessa opittuja arvoja.
2. Kytke gradienttien laskenta pois. Arvioinnissa niitä ei tarvita, ja ilman niitä
   laskenta on nopeampaa ja vie vähemmän muistia.
3. Laske, kuinka monta ennustetta osui oikein.

Funktion runko on samanlainen kuin `train_one_epoch`.

<details>
<summary>Vihje</summary>

- Arviointitila: `model.eval()`
- Gradientit pois: `with torch.no_grad():`
- Ennustettu luokka on se, jolla on suurin arvo: `outputs.argmax(1)`
- Oikeiden määrä batchissa: `(predicted == labels).sum().item()`

</details>

In [ ]:
def evaluate(model, loader):
    # TODO 1: aseta malli arviointitilaan

    running_loss = 0.0
    correct = 0
    total = 0

    # TODO 2: kytke gradienttien laskenta pois (with-lause)
    #         ja siirrä alla oleva silmukka sen sisään
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)

        # TODO 3: laske oikein menneiden ennusteiden määrä ja lisää se correct-muuttujaan

        total += labels.size(0)

    return running_loss / total, 100 * correct / total

<details>
<summary>Malliratkaisu (avaa vasta, kun olet yrittänyt itse)</summary>

```python
def evaluate(model, loader):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)

            predicted = outputs.argmax(1)
            correct += (predicted == labels).sum().item()

            total += labels.size(0)

    return running_loss / total, 100 * correct / total
```

</details>

### Tarkistus 6

Kokeillaan `evaluate`-funktiota kouluttamattomalla mallilla. Koska luokkia on 10,
satunnaisesti arvaavan mallin tarkkuuden pitäisi olla noin 10 %.

Kokeillaan myös tarkistuksessa 5 koulutettua mallia sillä yhdellä batchilla, jonka
se opetteli ulkoa. Sen tarkkuuden pitäisi olla lähes 100 %.

In [ ]:
torch.manual_seed(SEED)
untrained_model = MyCNN().to(device)

val_loss, val_acc = evaluate(untrained_model, valloader)
print(f"Kouluttamaton malli: loss {val_loss:.3f}, accuracy {val_acc:.2f}%")

check(not untrained_model.training, "evaluate asettaa mallin arviointitilaan")
check(0 < val_acc < 25, "Kouluttamattoman mallin tarkkuus on lähellä arvausta")

one_batch = [(images.cpu(), labels.cpu())]
_, batch_acc = evaluate(test_model, one_batch)
print(f"Ulkoa opeteltu batch: accuracy {batch_acc:.2f}%")

check(batch_acc > 90, "Ulkoa opeteltu batch tunnistetaan lähes täydellisesti")

## Koulutus

Kaikki osat ovat nyt valmiina. Seuraava solu kouluttaa mallin sinun kirjoittamillasi
funktioilla.

Asetukset ovat samat kuin harjoituksessa 2 (AdamW + OneCycleLR), mutta epocheja on
15, jotta koulutus on hieman nopeampi.

Paras malli tallennetaan tiedostoon `my_best_model.pt`.

In [ ]:
EPOCHS = 15

torch.manual_seed(SEED)
model = MyCNN().to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=5e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.003,
    epochs=EPOCHS,
    steps_per_epoch=len(trainloader)
)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
training_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    train_loss, train_acc = train_one_epoch(model, trainloader, optimizer, scheduler)
    val_loss, val_acc = evaluate(model, valloader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "my_best_model.pt")
        marker = " *"

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f}, acc: {train_acc:5.2f}% | "
        f"Val loss: {val_loss:.4f}, acc: {val_acc:5.2f}% | "
        f"{time.time() - epoch_start:4.1f} s{marker}"
    )

print()
print(f"Training time: {(time.time() - training_start) / 60:.1f} min")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

In [ ]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Train")
plt.plot(epochs_range, history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Train")
plt.plot(epochs_range, history["val_acc"], label="Validation")
plt.title("Accuracy (%)")
plt.xlabel("Epoch")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Lopullinen testaus

Ladataan paras malli ja arvioidaan se testidatalla. Kerätään samalla kaikki
ennusteet seuraavaa tehtävää varten.

In [ ]:
model.load_state_dict(torch.load("my_best_model.pt", map_location=device))

test_loss, test_acc = evaluate(model, testloader)
print(f"Test accuracy: {test_acc:.2f}%")

model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images.to(device, non_blocking=True))
        all_predictions.append(outputs.argmax(1).cpu())
        all_labels.append(labels)

all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)

## Tehtävä 7: Confusion matrix

Harjoituksessa 2 confusion matrix oli valmiina. Nyt lasket sen itse.

Confusion matrix on 10 × 10 -taulukko, jossa rivi on kuvan oikea luokka, sarake on
mallin ennustama luokka ja solun arvo kertoo, kuinka monta kertaa näin kävi.

Esimerkiksi jos oikea luokka on `3` (cat) ja ennuste `5` (dog), kasvatetaan solun
`confusion[3, 5]` arvoa yhdellä.

### Tehtävä

1. Täytä `confusion` käymällä läpi kaikki (oikea luokka, ennuste) -parit.
2. Laske `class_accuracy`: jokaisen luokan tarkkuus prosentteina.

Luokan tarkkuus = oikein ennustetut / kaikki kyseisen luokan kuvat × 100

<details>
<summary>Vihje</summary>

- Kaksi listaa voi käydä läpi rinnakkain `zip`-funktiolla:
  `for true_label, predicted in zip(all_labels.tolist(), all_predictions.tolist()):`
- Oikein ennustetut ovat taulukon diagonaalilla: `confusion.diag()`
- Rivin summa saadaan `confusion.sum(dim=1)`

</details>

In [ ]:
num_classes = len(classes)
confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)

# TODO 1: täytä confusion-taulukko


# TODO 2: laske jokaisen luokan tarkkuus prosentteina (tensori, jossa 10 arvoa)
class_accuracy = None

for name, acc in zip(classes, class_accuracy):
    print(f"{name:>10}: {acc:5.1f}%")

<details>
<summary>Malliratkaisu (avaa vasta, kun olet yrittänyt itse)</summary>

```python
num_classes = len(classes)
confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)

for true_label, predicted in zip(all_labels.tolist(), all_predictions.tolist()):
    confusion[true_label, predicted] += 1

class_accuracy = 100 * confusion.diag() / confusion.sum(dim=1)

for name, acc in zip(classes, class_accuracy):
    print(f"{name:>10}: {acc:5.1f}%")
```

</details>

### Tarkistus 7

In [ ]:
check(confusion.sum().item() == len(testset), "Taulukossa on kaikki 10 000 testikuvaa")
check(
    torch.equal(confusion.sum(dim=1), torch.bincount(all_labels, minlength=num_classes)),
    "Rivisummat vastaavat luokkien kokoja"
)
check(
    confusion.diag().sum().item() == (all_predictions == all_labels).sum().item(),
    "Diagonaalilla ovat oikeat ennusteet"
)
check(len(class_accuracy) == num_classes, "class_accuracy sisältää 10 arvoa")
check(
    abs(class_accuracy.float().mean().item() - test_acc) < 0.5,
    "Luokkien tarkkuuksien keskiarvo vastaa test accuracya"
)

In [ ]:
plt.figure(figsize=(9, 8))
plt.imshow(confusion, cmap="Blues")
plt.colorbar()

plt.xticks(range(num_classes), classes, rotation=45, ha="right")
plt.yticks(range(num_classes), classes)
plt.xlabel("Ennustettu luokka")
plt.ylabel("Oikea luokka")
plt.title("Confusion matrix (testidata)")

threshold = confusion.max() / 2

for row in range(num_classes):
    for col in range(num_classes):
        value = confusion[row, col].item()
        plt.text(
            col, row, value,
            ha="center", va="center",
            color="white" if value > threshold else "black",
            fontsize=8
        )

plt.tight_layout()
plt.show()

### Pohdittavaa

Kirjoita vastauksesi alla olevaan soluun.

1. Mikä luokka on mallille helpoin ja mikä vaikein?
2. Mitkä kaksi luokkaa sekoittuvat keskenään useimmin? Miksi luulet, että juuri ne?
3. Katso oppimiskäyriä. Onko mallissa merkkejä ylisovittamisesta? Mistä sen näkee?

Vastaukset:

1.
2.
3.

## Tehtävä 8: Haaste – paranna mallia

### Tavoite

Paranna mallin validointitarkkuutta vähintään yhdellä prosenttiyksiköllä yllä
saadusta tuloksesta. Jos haluat kovemman haasteen, tavoittele yli 90 %.

Käytä vain validointidataa muutosten vertailuun. Aja testidata vasta, kun olet
valinnut lopullisen mallisi.

### Ideoita

Malli:

- Lisää neljäs lohko tai enemmän kanavia
- Kokeile eri Dropout-arvoja
- Kokeile `nn.AdaptiveAvgPool2d(1)` ennen luokittelijaa ison `Linear`-kerroksen sijaan

Koulutus:

- Enemmän epocheja
- Eri `max_lr`-arvo
- `nn.CrossEntropyLoss(label_smoothing=0.1)`
- SGD-optimointi: `optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)`

Data:

- Lisää augmentaatioita, esim. `transforms.ColorJitter` tai `transforms.RandomErasing`
  (huom. `RandomErasing` toimii tensorilla, joten se tulee `ToTensor()`-muunnoksen jälkeen)

Nopeus:

- Mixed precision: `torch.autocast(device_type="cuda", dtype=torch.bfloat16)`
- Suurempi batch size

Muuta yhtä asiaa kerrallaan, muuten et tiedä, mikä muutos auttoi.

### Tehtävä

Kirjoita parannettu mallisi ja koulutuksesi alla oleviin soluihin. Voit kopioida
pohjaksi koodia yllä olevista soluista.

Jos muutat muunnoksia (`train_transform`), muista luoda datasetit ja DataLoaderit
uudelleen.

In [ ]:
# Kirjoita parannettu mallisi tähän

In [ ]:
# Kouluta ja arvioi parannettu mallisi tähän

### Kokeilupäiväkirja

Kirjaa kokeilusi taulukkoon. Näin näet, mikä toimi ja mikä ei.

| Kokeilu | Muutos | Val accuracy | Aika / epoch |
| --- | --- | --- | --- |
| 0 | Lähtötaso (tehtävät 1–7) | | |
| 1 | | | |
| 2 | | | |
| 3 | | | |

Lopullinen test accuracy:

Mikä muutos auttoi eniten?

## Yhteenveto

Tässä harjoituksessa kirjoitit itse koneoppimisen työketjun keskeiset osat:

| Tehtävä | Mitä opit? |
| --- | --- |
| 1 | Datan tilastojen laskeminen NumPyllä |
| 2 | Normalisoinnin ja augmentaation määrittely |
| 3 | Datan jakaminen ilman vuotoa osien välillä |
| 4 | Oman neuroverkon rakentaminen PyTorchilla |
| 5 | Koulutusaskeleen viisi vaihetta ja "overfit one batch" -testi |
| 6 | Mallin oikeaoppinen arviointi |
| 7 | Confusion matrixin laskeminen ja tulkinta |
| 8 | Mallin kehittäminen kokeilemalla |

Tehtävän 5 koulutusaskel toistuu lähes samanlaisena kaikessa PyTorch-koulutuksessa,
oli kyse pienestä CNN:stä tai suuresta kielimallista.

Muista lopuksi sulkea Jupyter-sessio, jotta varatut laskentaresurssit vapautuvat:

My Interactive Sessions → Oma Jupyter session → Cancel